# Notebook 04: PostgreSQL load and SQL analysis

**Project:** Global Layoffs Intelligence · **Phase:** 4 of 6
**Input:** the verified files in `data/processed/` (Phase 2). **Output:** the database tables in `layoffs_db` (schemas `staging`, `core`, `audit`), plus query results in `reports/phase4/`.

The real work is done by tested Python modules, so this notebook stays short and readable:

| Module | What it does |
|---|---|
| `src/phase4_load.py` | pre-flight checks, then staging, then core, in **one transaction**, then integrity checks and the audit trail |
| `src/phase4_checks.py` | about 140 checks comparing the database with the CSV files |
| `src/phase4_analytics.py` | runs `sql/analytics/*.sql` and compares the results with the Phase 3 chart data |

**Before you run it:**
1. Install PostgreSQL.
2. Create the database (`sql/README.md`, step 2).
3. Copy `.env.example` to `.env` and fill it in.
4. Use the *Python (layoffs-env)* kernel and run notebooks 01 to 03 first.

**This notebook writes to the database**, but only to its own tables. Rerunning it is safe.

## 1. Connect to PostgreSQL

**Objective.** Check that the database settings work before loading anything.

**Concept.** The password is never written in code. It comes from `.env` or environment variables (`PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER`, `PGPASSWORD`).
If the connection fails, the error says whether the problem is the **environment** (server not running, wrong password) and not the pipeline.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "data" / "raw").is_dir():
            return folder
    raise FileNotFoundError("Start Jupyter inside the project folder.")


PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import phase4_config as config
import phase4_load as loader
import phase4_analytics as analytics

conn = config.connect(root=PROJECT_ROOT)


def sql_df(query, params=None):
    # run a read-only query and return a DataFrame
    with conn.cursor() as cur:
        cur.execute(query, params)
        out = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
    conn.rollback()
    return out


print("connected to", config.describe_target())
sql_df("SELECT current_setting('server_version') AS server_version, current_setting('server_encoding') AS encoding")

connected to postgresql://layoffs_app@localhost:5433/layoffs_db


,server_version,encoding
0,16.15 (Ubuntu 16.15-0ubuntu0.24.04.1),UTF8


## 2. Load the processed files

**Objective.** Copy the 10 verified CSV files into PostgreSQL without changing a value.

**Concept.** One run has five stages:
1. **Pre-flight**, in Python, with no database writes. It checks the headers, the types of every value, the primary keys, and the SHA-256 of each file against the Phase 2 manifest.
2. **Schema.** It creates the tables if they are missing. It never drops or alters anything.
3. **Staging.** It copies each CSV as plain text.
4. **Core.** It converts the text to the right types (dates, integers, booleans) and replaces the core tables.
5. **Validation.** It runs about 140 checks against the CSV files.

Stages 3 to 5 are **one transaction**: if anything fails, PostgreSQL rolls everything back and the previous data stays as it was.

In [2]:
summary = loader.run_load()

1/5 pre-flight: validating 10 CSV files in data/processed/ ...


    OK: 13,794 rows, headers/types/keys valid, hashes match the Phase 2 manifest
2/5 schema: applying sql/ddl (create-if-not-exists only) ...
    OK: load run 4 recorded in audit.load_runs
3/5 staging: COPY into staging tables ...


4/5 core: replacing core tables inside one transaction ...


5/5 validation: running integrity and reconciliation checks ...


    OK: 142 checks passed; committed (run 4).


In [3]:
counts = pd.Series(summary["core_counts"], name="rows").to_frame()
counts["csv_rows"] = [None if t == "source_datasets" else
                      next(r.csv_rows for r in summary["files"] if r.core_table == t) for t in counts.index]
assert all(counts["rows"].iloc[1:] == counts["csv_rows"].iloc[1:])      # every table = its CSV
counts

,rows,csv_rows
source_datasets,2,NaN
company_name_mapping,3532,3532.0
industry_mapping,77,77.0
layoff_events,5035,5035.0
layoff_event_departments,70,70.0
quality_issues,4886,4886.0
transformation_log,16,16.0
field_lineage,54,54.0
cross_source_industry_disagreements,9,9.0
possible_duplicate_pairs,18,18.0


## 3. The load audit trail

**Objective.** Show what the database recorded about each load.

**Concept.** `audit.load_runs` has one row per attempt, **including failed ones**. `audit.load_files` stores each file's fingerprint and row counts at every stage.
This is how you can prove later exactly which files were loaded and when.

In [4]:
sql_df("SELECT run_id, started_at, finished_at, status, error_stage, manifest_checked, input_dir "
       "FROM audit.load_runs ORDER BY run_id DESC LIMIT 5")

,run_id,started_at,finished_at,status,error_stage,manifest_checked,input_dir
0,4,2026-10-23 13:23:20.768032+00:00,2026-10-23 13:23:21.985226+00:00,succeeded,NaN,True,data/processed
1,3,2026-10-23 13:13:52.869263+00:00,2026-10-23 13:13:54.416093+00:00,succeeded,NaN,True,data/processed
2,2,2026-10-23 13:13:50.531047+00:00,2026-10-23 13:13:52.036235+00:00,succeeded,NaN,True,data/processed
3,1,2026-10-23 13:13:37.751245+00:00,2026-10-23 13:13:39.306630+00:00,failed,validation,True,data/processed


In [5]:
run_id = summary["run_id"]
files = sql_df("SELECT file_name, left(sha256, 12) AS sha256, matches_manifest, csv_rows, staging_rows, core_rows "
               "FROM audit.load_files WHERE run_id = %s ORDER BY file_name", (run_id,))
assert files["matches_manifest"].all() and (files["csv_rows"] == files["core_rows"]).all()
files

,file_name,sha256,matches_manifest,csv_rows,staging_rows,core_rows
0,company_name_mapping.csv,54be3d99e71a,True,3532,3532,3532
1,cross_source_industry_disagreements.csv,5a375a3ffa7e,True,9,9,9
2,excluded_candidate_pairs.csv,97e9fd91b926,True,97,97,97
3,field_lineage.csv,3b84aeec32c0,True,54,54,54
4,industry_mapping.csv,1c1338df0810,True,77,77,77
5,layoff_event_departments.csv,08005235def7,True,70,70,70
6,layoff_events.csv,9a42f4f1a1a7,True,5035,5035,5035
7,possible_duplicate_pairs.csv,8a8379fe5fb3,True,18,18,18
8,quality_issues.csv,3159195d3bb7,True,4886,4886,4886
9,transformation_log.csv,365beee7cdc6,True,16,16,16


## 4. Integrity and reconciliation checks

**Objective.** Summarise the checks that ran inside the load transaction.

**Concept.** Each check compares the database with a value computed **separately** from the CSV files.
- **Keys:** every key is unique, and every child row points to a real event.
- **Nulls:** every missing value is still missing (never 0).
- **Flags:** TRUE, FALSE and *not computed* are all kept.
- **Sums:** headcount totals are exact.
- **Distributions:** Unknown and Unresolved are still there.

The checks expect the data's **real** gaps: they never require all headcounts or countries to be known.

In [6]:
checks = sql_df("SELECT category, count(*) AS checks, count(*) FILTER (WHERE passed) AS passed "
                "FROM audit.validation_results WHERE run_id = %s GROUP BY 1 ORDER BY 1", (run_id,))
assert (checks["checks"] == checks["passed"]).all()
checks

,category,checks,passed
0,bridge,5,5
1,counts,21,21
2,dates,3,3
3,distributions,10,10
4,duplicates,4,4
5,keys,11,11
6,mappings,7,7
7,nulls,49,49
8,qc_flags,21,21
9,reconciliation,8,8


In [7]:
sql_df("SELECT check_id, expected, actual, passed FROM audit.validation_results WHERE run_id = %s AND check_id IN "
       "('nulls.employees_laid_off', 'recon.employees_laid_off', 'recon.workforce_pct', 'dates.suspect_preserved', "
       "'dup.group_members', 'bridge.weights_sum_to_1', 'keys.fk.layoff_event_departments.event_key')", (run_id,))

,check_id,expected,actual,passed
0,bridge.weights_sum_to_1,0,0,True
1,dates.suspect_preserved,T-00002|2025-01-09=1; T-00007|2025-02-05=1; T-...,T-00002|2025-01-09=1; T-00007|2025-02-05=1; T-...,True
2,dup.group_members,DG-001=3; DG-002=2; DG-003=2; DG-004=2; DG-005...,DG-001=3; DG-002=2; DG-003=2; DG-004=2; DG-005...,True
3,keys.fk.layoff_event_departments.event_key,0,0,True
4,nulls.employees_laid_off,fliar_global_sample=2511; tech_layoffs_2025_20...,fliar_global_sample=2511; tech_layoffs_2025_20...,True
5,recon.employees_laid_off,fliar_global_sample|2489=2788908; tech_layoffs...,fliar_global_sample|2489=2788908; tech_layoffs...,True
6,recon.workforce_pct,fliar_global_sample|1121=24761.242033; tech_la...,fliar_global_sample|1121=24761.242033; tech_la...,True


## 5. What the core tables look like

**Objective.** Look at the schema and confirm that missing values arrived as SQL `NULL`.

**Concept.** `NULL` means "unknown". `count(column)` skips NULLs, so `count(*) - count(column)` gives the number of missing values.
If a missing headcount had been turned into 0, the sum would not change, but the number of known values would, and so would every average.

In [8]:
sql_df("SELECT column_name, data_type, is_nullable FROM information_schema.columns "
       "WHERE table_schema = 'core' AND table_name = 'layoff_events' ORDER BY ordinal_position").head(20)

,column_name,data_type,is_nullable
0,event_key,text,NO
1,source_dataset,text,NO
2,source_confidence,text,NO
3,source_file,text,NO
4,source_row_number,integer,NO
5,source_event_id,text,YES
6,company_name_raw,text,NO
7,company_name,text,NO
8,company_name_method,text,NO
9,company_name_review_status,text,NO


In [9]:
nulls = sql_df('''SELECT source_dataset,
                         count(*)                               AS rows,
                         count(*) - count(employees_laid_off)   AS headcount_missing,
                         count(*) FILTER (WHERE employees_laid_off = 0) AS headcount_zero,
                         count(*) - count(workforce_pct)        AS pct_missing,
                         count(*) - count(country)              AS country_missing,
                         count(*) FILTER (WHERE qc_headcount_not_verified IS NULL) AS flag_not_computed
                  FROM core.layoff_events GROUP BY 1 ORDER BY 1''')
assert nulls["headcount_zero"].sum() == 0 and nulls["headcount_missing"].sum() == 2511
nulls

,source_dataset,rows,headcount_missing,headcount_zero,pct_missing,country_missing,flag_not_computed
0,fliar_global_sample,5000,2511,0,3879,470,0
1,tech_layoffs_2025_2026,35,0,0,0,0,35


## 6. Analytical SQL queries

**Objective.** Run the main business questions in SQL.

**Concept.** Every file in `sql/analytics/` starts with a header that states its question, its **counting unit**, its **filters**, how missing values are treated, and which Phase 3 file it reconciles with.
Global figures always use `core.v_global_events` (5,000 rows). The 35 tech rows have their own queries and are never added in.

In [10]:
def run_sql_file(name):
    path = PROJECT_ROOT / "sql" / "analytics" / f"{name}.sql"
    header = [l[3:] for l in path.read_text().splitlines() if l.startswith("-- ") and ":" in l][:3]
    print("\n".join(header))
    return analytics.run_query(conn, path)[0]


run_sql_file("02_yearly_events_and_known_job_cuts")

Question  : Per year, how many events, and how many jobs were cut where the headcount is known?
Unit      : one row = one calendar year (global sample).
Filters   : global sample only; all rows counted (furloughs and duplicate candidates included - see 19).


,event_year,events,events_with_headcount,verified_jobs_cut,median_cut,headcount_coverage_pct,share_of_events_pct,permanent_events,furlough_events,partial_year,period_note
0,2019,476,238,247634,200.0,50.0,9.5,472,4,False,complete year
1,2020,968,473,751961,300.0,48.9,19.4,863,105,False,complete year
2,2021,265,111,152748,210.0,41.9,5.3,258,7,False,complete year
3,2022,525,240,193251,142.0,45.7,10.5,523,2,False,complete year
4,2023,885,448,423496,169.0,50.6,17.7,882,3,False,complete year
5,2024,791,405,434305,175.0,51.2,15.8,787,4,False,complete year
6,2025,762,407,416695,200.0,53.4,15.2,754,8,False,complete year
7,2026,328,167,168818,153.0,50.9,6.6,328,0,True,partial year: Jan to 14 Jun 2026; May-Jun poss...


In [11]:
run_sql_file("03_industry_ranking")[["rank_by_events", "industry_group", "events", "share_of_events_pct",
                                     "headcount_coverage_pct"]]

Question  : Which industry groups have the most layoff events?
Unit      : one row = one industry group (17 values including 'Unknown').
Filters   : global sample only; all rows counted.


,rank_by_events,industry_group,events,share_of_events_pct,headcount_coverage_pct
0,1,Unknown,1564,31.3,43.5
1,2,Retail & Consumer,680,13.6,53.4
2,3,Finance & Fintech,375,7.5,55.5
3,4,Healthcare & Life Sciences,298,6.0,53.0
4,5,Software & IT,281,5.6,54.1
5,6,Food & Agriculture,268,5.4,50.4
6,7,Media & Gaming,223,4.5,47.5
7,8,Industrial & Manufacturing,208,4.2,57.2
8,9,Energy & Resources,191,3.8,51.8
9,10,Automotive,185,3.7,56.2


In [12]:
countries = run_sql_file("06_country_comparison")
countries[countries["country_status"] != "ok"]          # missing and unresolved values stay visible

Question  : Which countries have the most events, and which country values are missing or unresolved?
Unit      : one row = one event-country value (country is where the layoff happened, not the HQ).
Filters   : global sample only; all rows counted.


,country,region,events,events_with_headcount,verified_jobs_cut,median_cut,headcount_coverage_pct,share_of_events_pct,country_status
2,Unknown,Unknown,470,161,265700.0,150.0,34.3,9.4,missing
81,Arizona,Unresolved,1,0,NaN,NaN,0.0,0.0,not a country (flagged)
95,Nevada,Unresolved,1,1,100.0,100.0,100.0,0.0,not a country (flagged)
96,New York,Unresolved,1,0,NaN,NaN,0.0,0.0,not a country (flagged)
101,Quebec,Unresolved,1,0,NaN,NaN,0.0,0.0,not a country (flagged)
102,Rhode Island,Unresolved,1,1,125.0,125.0,100.0,0.0,not a country (flagged)
104,Sa,Unresolved,1,0,NaN,NaN,0.0,0.0,not a country (flagged)


In [13]:
run_sql_file("09_workforce_pct_distribution")

Question  : What share of the workforce do layoffs cut?
Unit      : one row = one band of workforce_pct (lower bound included, upper excluded; last band 50-100%).
Filters   : global sample; ONLY rows with a known workforce_pct (1,121 of 5,000). Unknown percentages are


,workforce_pct_band,events,share_of_known_pct,denominator_rows_with_pct,rows_without_pct_excluded
0,0-5%,94,8.4,1121,3879
1,5-10%,190,16.9,1121,3879
2,10-20%,360,32.1,1121,3879
3,20-30%,177,15.8,1121,3879
4,30-50%,158,14.1,1121,3879
5,50-100%,142,12.7,1121,3879


### 6.1 Repeat layoffs by company (new in Phase 4)

**Concept.** A company with 2 or more global events is a *repeat* company. Some of those events may be one event reported twice (rows flagged `possible_internal_duplicate`), so the second query shows the count both ways.
Nothing is removed: the *what-if* column only ignores the flagged rows.

In [14]:
repeat = run_sql_file("15_repeat_layoff_companies")
print(f"{len(repeat)} companies with 2+ events")
repeat.head(10)[["company_name", "events", "events_flagged_internal_duplicate", "distinct_years",
                 "first_event_date", "last_event_date", "known_jobs_cut", "name_needs_review"]]

Question  : Which companies had more than one layoff event?
Unit      : one row = one company_name (cleaned name, Phase 2) with 2 or more global events.
Filters   : global sample only. ALL rows are counted; how many of them are flagged as possible internal


688 companies with 2+ events


,company_name,events,events_flagged_internal_duplicate,distinct_years,first_event_date,last_event_date,known_jobs_cut,name_needs_review
0,Microsoft,22,0,8,2019-07-28,2026-06-10,51089.0,False
1,Boeing,19,0,7,2019-01-21,2026-02-05,116570.0,False
2,Intel,19,0,7,2019-03-26,2026-02-06,90727.0,False
3,Bosch,18,4,7,2019-01-07,2026-04-19,40836.0,False
4,Nestle,18,0,8,2019-01-20,2026-04-21,17422.0,False
5,Walmart,18,0,7,2019-02-28,2026-03-31,9099.0,False
6,Novartis,16,4,6,2019-10-07,2026-03-30,3208.0,False
7,Nissan,15,0,6,2019-03-05,2026-05-05,66849.0,False
8,AT&T,14,0,7,2019-01-07,2025-12-22,33955.0,False
9,Oracle,14,0,6,2019-03-25,2026-06-02,58097.0,False


In [15]:
run_sql_file("16_repeat_layoff_distribution")

Question  : How many companies had 1, 2, 3 or 4+ layoff events - and does that change if possible
Unit      : one row = one bucket of "events per company"; companies = number of distinct company_name.
Filters   : global sample only. Column companies_all counts every row. companies_excl_internal_dups is a


,events_per_company,companies_all,companies_excl_internal_dups
0,0 (every row flagged),0,9
1,1,2779,2778
2,2,401,397
3,3,126,125
4,4+,161,158


### 6.2 Sensitivity and the tech file (kept separate)

In [16]:
run_sql_file("19_sensitivity_by_year")

Question  : Do furloughs or unresolved duplicate candidates drive the yearly results?
Unit      : one row = one year.
Filters   : global sample only (tech rows never included); all_events counts every row.


,event_year,all_events,excl_furloughs,excl_internal_duplicates,excl_cross_source_duplicate_candidates,furlough_share_pct,known_jobs_all,known_jobs_excl_furloughs,known_jobs_excl_all_duplicate_flags,period_note
0,2019,476,472,471,476,0.8,247634,247514,246282,complete year
1,2020,968,863,962,968,10.8,751961,583793,749341,complete year
2,2021,265,258,260,265,2.6,152748,125998,152042,complete year
3,2022,525,523,521,525,0.4,193251,193251,193121,complete year
4,2023,885,882,879,880,0.3,423496,423199,396154,complete year
5,2024,791,787,777,789,0.5,434305,433380,428445,complete year
6,2025,762,754,760,753,1.0,416695,415721,360705,complete year
7,2026,328,328,328,326,0.0,168818,168818,145318,partial year


In [17]:
run_sql_file("12_tech_ai_related_summary")             # LOWER-CONFIDENCE TECH FILE ONLY

Question  : How do events marked AI_Related = Yes / Partial / No compare?
Unit      : one row = one AI_Related value, as reported by the file (no stated definition).
Filters   : core.v_tech_events only. suspect_dates counts rows flagged qc_suspect_date (6 in total);


,ai_related,events,reported_laid_off,median_pct_workforce,median_severance_weeks,suspect_dates
0,Yes,9,21720,5.5,12.0,1
1,Partial,6,48772,8.0,11.0,1
2,No,20,83030,15.0,12.0,4


## 7. Reconcile every query with Phase 3

**Objective.** Prove that SQL gives the same answers as the pandas analysis in notebook 03.

**Concept.** Each query's result is compared **cell by cell** with the matching chart-data file in `reports/eda/`.
Rounded percentages may differ by one rounding step, because pandas and PostgreSQL round halves differently. Every other value must be identical.
Queries with no Phase 3 equivalent (repeat layoffs, workforce % by year) are checked against pandas in `tests/test_phase4_db.py`.

In [18]:
recon = analytics.run_all(verbose=False)
assert (recon["status"] == "ok").all() and (recon["reconciliation"] != "MISMATCH").all()
print(f"{len(recon)} queries ran, {int(recon['cells_compared'].sum())} cells compared with Phase 3, "
      f"{int(recon['mismatches'].sum())} differences")
recon[["query", "result_rows", "reconciliation", "rows_compared", "cells_compared", "mismatches"]]

19 queries ran, 979 cells compared with Phase 3, 0 differences


,query,result_rows,reconciliation,rows_compared,cells_compared,mismatches
0,sql/analytics/01_monthly_events.sql,90,MATCH,90,180,0
1,sql/analytics/02_yearly_events_and_known_job_c...,8,MATCH,8,72,0
2,sql/analytics/03_industry_ranking.sql,17,MATCH,17,119,0
3,sql/analytics/04_industry_share_by_year.sql,135,MATCH,136,136,0
4,sql/analytics/05_region_comparison.sql,7,MATCH,7,42,0
5,sql/analytics/06_country_comparison.sql,111,MATCH,15,120,0
6,sql/analytics/07_us_share_by_year.sql,8,MATCH,8,32,0
7,sql/analytics/08_layoff_size_distribution.sql,5,MATCH,5,10,0
8,sql/analytics/09_workforce_pct_distribution.sql,6,MATCH,6,6,0
9,sql/analytics/10_workforce_pct_by_year.sql,9,not applicable (checked in tests),0,0,0


In [19]:
conn.close()

## 8. What this phase established

- All 10 processed files are in PostgreSQL. Every row count equals its CSV, and every key, relationship and missing value is preserved. The checks are stored in `audit.validation_results`.
- Loading is **repeatable**: a rerun replaces the data with identical rows, and a failed run changes nothing (tested in `tests/test_phase4_db.py`).
- The SQL results equal the Phase 3 results.
- **The data's limitations are still there on purpose.** About 50% of headcounts are missing, 31% of sectors are Unknown, 6 countries are unresolved, the duplicates are unresolved, 2026 is partial, and May-June 2026 is probably incomplete.

**Next (only after your review):** Phase 5, the layoff-risk model.